# OGGM-VAS with RGI 7.0: CRU reconstruction of all regions (no projections)

**What this notebook does**
1. Loads each RGI 7.0 region, prepares the glaciers (rgitopo directories, masks, CRU climate, VAS calibration).
2. Runs VAS from 1901 to 2019 forced by **CRU only** (the reconstruction). No CMIP data, no projections.
3. Aggregates to regional volume / area / length / mass loss / sea-level equivalent, plots them, saves CSVs.

**Kernel:** needs OGGM >= 1.6 (e.g. *Python (OGGM RGI7 + VAS)*); cell 1 stops with a clear message otherwise.

**How to run:** edit the config (cell 2) if you want, then run all cells. Cells 1-11 only define things; the **last cell** starts the calculations, either for all 19 RGI 7.0 regions one after the other or for a single region (switch at the top of that cell).

Every region gets its own folder in `WORK_ROOT` (the same folders as the CMIP notebook, so downloaded glacier directories, CRU climate and calibration are reused). Overall progress is in `all_regions_cru_log.csv` and `all_regions_reconstruction.csv`. Everything is re-runnable: finished steps and finished regions are detected on disk and skipped.

In [ ]:
# ======================================================================
# 1. IMPORTS AND OGGM 1.6.3 / VAS COMPATIBILITY PATCHES
# ======================================================================
import json
import os
import re
import shutil
import time
from pathlib import Path

import numpy as np

# --- NumPy 2 compatibility -------------------------------------------------------------------
# Older OGGM / OGGM-VAS code still uses names that NumPy 2.0 removed (np.NaN, np.Inf, ...), which made
# every glacier fail in the reconstruction run. Re-create the removed aliases (only if missing).
for _old, _new in {"NaN": np.nan, "Inf": np.inf, "infty": np.inf, "PINF": np.inf, "NINF": -np.inf,
                   "float_": np.float64, "complex_": np.complex128, "product": np.prod,
                   "cumproduct": np.cumprod, "sometrue": np.any, "alltrue": np.all}.items():
    if not hasattr(np, _old):
        setattr(np, _old, _new)
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt

import oggm.cfg as cfg
from oggm import utils, workflow, tasks
from oggm import GlacierDirectory

# --- old VAS climate_info basename -----------------------------------------
if "climate_info" not in cfg.BASENAMES:
    cfg.BASENAMES["climate_info"] = (
        "climate_info.json",
        "VAS glacier climate and calibration information.",
    )


# --- JSON compatibility patch for OGGM 1.6.3 + VAS --------------------------
def write_json_compat(self, var, filename, filesuffix=""):
    def np_convert(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

    with open(self.get_filepath(filename, filesuffix=filesuffix), "w") as f:
        json.dump(var, f, default=np_convert)


GlacierDirectory.write_json = write_json_compat

# --- get_climate_info compatibility patch (merge old VAS climate_info.json) -
_original_get_climate_info = GlacierDirectory.get_climate_info


def get_climate_info_compat(self, input_filesuffix=""):
    out = _original_get_climate_info(self, input_filesuffix=input_filesuffix)
    try:
        vas_info = self.read_json("climate_info")
        if isinstance(vas_info, dict):
            out.update(vas_info)
    except FileNotFoundError:
        pass
    return out


GlacierDirectory.get_climate_info = get_climate_info_compat

import oggm_vas as vascaling

import oggm
_oggm_version = tuple(int(p) for p in re.findall(r"\d+", oggm.__version__)[:2])
if _oggm_version < (1, 6):
    raise RuntimeError(
        f"OGGM {oggm.__version__} is too old for RGI 7.0 (needs >= 1.6). "
        "Switch to the kernel with OGGM 1.6.x (e.g. 'Python (OGGM RGI7 + VAS)').")

In [ ]:
# ======================================================================
# 2. CONFIGURATION
# ======================================================================
# ------------------------------------------------------------
# Region (RGI 7.0): 1-19
# ------------------------------------------------------------
RGI_VERSION = "70G"
RGI_REGION = 11

RGI_REGION_NAMES = {
    1: "Alaska", 2: "Western Canada and USA", 3: "Arctic Canada North",
    4: "Arctic Canada South", 5: "Greenland Periphery", 6: "Iceland",
    7: "Svalbard", 8: "Scandinavia", 9: "Russian Arctic", 10: "North Asia",
    11: "Central Europe", 12: "Caucasus and Middle East", 13: "Central Asia",
    14: "South Asia West", 15: "South Asia East", 16: "Low Latitudes",
    17: "Southern Andes", 18: "New Zealand", 19: "Antarctic and Subantarctic",
}

# ------------------------------------------------------------
# Reconstruction period (forced by CRU, no projections)
# ------------------------------------------------------------
MODEL_START_YEAR = 1901
CRU_RECON_END_YEAR = 2019
ANOMALY_REFERENCE_PERIOD = ("1961", "1990")   # only for the anomalies in the regional CRU climate CSV

# ------------------------------------------------------------
# VAS calibration table (RGI6/CRU reference table shipped with OGGM-VAS)
# ------------------------------------------------------------
VAS_REFERENCE_TSTARS_FILE = "vas_ref_tstars_rgi6_cru4.csv"

# ------------------------------------------------------------
# Folders
# ------------------------------------------------------------
WORK_ROOT = Path(os.environ.get("OGGM_WORK_ROOT", "~/oggm_work")).expanduser()   # one sub-folder per region
# (on a server: export OGGM_WORK_ROOT=/big/disk/oggm_work before starting; no edit needed)
WORKDIR = WORK_ROOT / f"vas_rgi70_region{RGI_REGION:02d}"    # (set per region by the run cell)

# ------------------------------------------------------------
# Run control
# ------------------------------------------------------------
# DEM used for the topography. None = automatic (OGGM default): NASADEM between 56S and 60N,
# COPDEM90 elsewhere. Or force one, e.g. "COPDEM90" / "COPDEM30" / "NASADEM" (must exist in the folders).
DEM_SOURCE = None

MAX_GLACIERS = None          # None = all glaciers. Use e.g. 20 for a quick test run!
SKIP_FINISHED_REGIONS = True # True: a region that finished with the same settings is skipped
RECOMPUTE_BASE = False       # True: redo masks / CRU climate / calibration / reconstruction run

# ------------------------------------------------------------
# Disk space
# ------------------------------------------------------------
# Per-glacier time series (volume/area/length of every glacier, every year).
# The regional CSVs and plots do NOT depend on this file.
#   "parquet": compact (float32, compressed; needs pyarrow, otherwise falls back to "csv.gz")
#   "csv.gz" : gzip-compressed CSV, no extra package needed
#   "csv"    : plain CSV (largest)
#   False    : do not save per-glacier results
SAVE_GLACIER_RESULTS = "parquet"

# True: after a region finished successfully, delete its OGGM glacier folders
# (the big per-glacier files in WORKDIR/per_glacier). Regional CSVs, plots and the
# "region finished" marker are kept, so restarts still skip the region. Re-running that
# region with different settings means downloading and calibrating it again.
CLEAN_GLACIER_DIRS = True

# ------------------------------------------------------------
# Physical constants
# ------------------------------------------------------------
RHO_ICE = 900.0
RHO_WATER = 1000.0
OCEAN_AREA_KM2 = 3.625e8

# ------------------------------------------------------------
# Derived settings and checks (do not edit)
# ------------------------------------------------------------
REGION_NAME = RGI_REGION_NAMES[RGI_REGION]
REGION_SLUG = REGION_NAME.replace(" ", "")

if RGI_REGION not in RGI_REGION_NAMES:
    raise ValueError("RGI_REGION must be between 1 and 19.")
if CRU_RECON_END_YEAR <= MODEL_START_YEAR:
    raise ValueError("CRU_RECON_END_YEAR must be after MODEL_START_YEAR.")
if SAVE_GLACIER_RESULTS not in ("parquet", "csv.gz", "csv", False):
    raise ValueError('SAVE_GLACIER_RESULTS must be "parquet", "csv.gz", "csv" or False.')

CRU_HIST_SUFFIX = f"_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Default region: RGI70-{RGI_REGION:02d} {REGION_NAME} (used when RUN_ALL_REGIONS = False)")
print(f"Reconstruction: CRU-forced, {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (no projections)")
print(f"Work root     : {WORK_ROOT}")
if MAX_GLACIERS is not None:
    print(f"TEST MODE     : only {MAX_GLACIERS} glaciers (set MAX_GLACIERS = None for the full run)")

In [ ]:
# ======================================================================
# 3. REGION HELPERS AND OGGM-VAS SETUP (functions only, nothing runs here)
# ======================================================================
_OGGM_READY = False


def set_region(region):
    """Switch all region-dependent settings (name, output folder)."""
    global RGI_REGION, REGION_NAME, REGION_SLUG, WORKDIR
    if region not in RGI_REGION_NAMES:
        raise ValueError(f"RGI region must be between 1 and 19, got {region}.")
    RGI_REGION = region
    REGION_NAME = RGI_REGION_NAMES[region]
    REGION_SLUG = REGION_NAME.replace(" ", "")
    WORKDIR = WORK_ROOT / f"vas_rgi70_region{region:02d}"
    WORKDIR.mkdir(parents=True, exist_ok=True)


def init_oggm():
    """Initialize OGGM-VAS once; afterwards only the working directory changes."""
    global _OGGM_READY
    if not _OGGM_READY:
        print("\nInitializing OGGM-VAS...")
        vascaling.initialize()

        cfg.PARAMS["rgi_version"] = RGI_VERSION
        cfg.PARAMS["baseline_climate"] = "CRU"      # reference climate (calibration + bias correction)
        cfg.PARAMS["hydro_month_nh"] = 1            # calendar years
        cfg.PARAMS["hydro_month_sh"] = 1
        cfg.PARAMS["use_multiprocessing"] = False
        cfg.PARAMS["use_intersects"] = False
        cfg.PARAMS["continue_on_error"] = True      # a failing glacier does not stop the run

        # VAS parameters
        cfg.PARAMS["prcp_scaling_factor"] = 3.0
        cfg.PARAMS["temp_melt"] = 0.0
        cfg.PARAMS["temp_all_solid"] = 4.0
        cfg.PARAMS["temp_all_liq"] = 2.0
        cfg.PARAMS["climate_qc_months"] = 3
        cfg.PARAMS["temp_default_gradient"] = -0.0065
        cfg.PARAMS["temp_local_gradient_bounds"] = [-0.009, -0.003]
        cfg.PARAMS["temp_use_local_gradient"] = False
        cfg.PARAMS["prcp_default_gradient"] = 0.0004
        cfg.PARAMS["mu_star_halfperiod"] = 15
        cfg.PARAMS["min_mu_star"] = 5
        cfg.PARAMS["max_mu_star"] = 10000
        cfg.PARAMS["use_bias_for_run"] = True       # use the calibrated VAS bias in model runs
        cfg.PARAMS["run_mb_calibration"] = False
        _OGGM_READY = True
    cfg.PATHS["working_dir"] = str(WORKDIR)

In [ ]:
# ======================================================================
# 4. LOAD RGI 7.0 REGION (function)
# ======================================================================
def find_rgi7_region_file(region):
    """Shapefile of one RGI 7.0 region.

    OGGM's own utils.get_rgi_region_file() fails here for regions 1-9 (assert len(f) == 1),
    so the file is looked up directly in OGGM's RGI folder by its exact name, e.g.
    RGI2000-v7.0-G-01_alaska.shp. OGGM's function is only the fallback.
    """
    letter = RGI_VERSION[-1]                              # "G" = glaciers, "C" = complexes
    try:
        rgi_dir = Path(utils.get_rgi_dir(version=RGI_VERSION))   # downloads/extracts if needed
    except Exception:
        rgi_dir = Path(cfg.PATHS["rgi_dir"])
    hits = sorted(p for p in rgi_dir.rglob(f"RGI2000-v7.0-{letter}-{region:02d}_*.shp")
                  if "intersect" not in p.name.lower())
    if len(hits) == 1:
        return hits[0]
    try:
        return Path(utils.get_rgi_region_file(region, version=RGI_VERSION))
    except Exception as exc:
        raise RuntimeError(
            f"Could not find the RGI 7 file of region {region:02d} in {rgi_dir} "
            f"(exact-name matches: {[str(h) for h in hits]}; OGGM: {type(exc).__name__}).") from exc


def load_region_glaciers():
    print(f"\nLoading RGI 7.0 region {RGI_REGION:02d}: {REGION_NAME}")

    # OGGM 1.6.x downloads the official RGI7 archive and extracts the region
    rgi_file = find_rgi7_region_file(RGI_REGION)
    print("RGI file:", rgi_file)

    rgidf = gpd.read_file(rgi_file)
    print("Number of RGI7 glaciers:", len(rgidf))

    if "rgi_id" not in rgidf.columns:
        if "RGIId" in rgidf.columns:
            rgidf["rgi_id"] = rgidf["RGIId"]
        else:
            raise RuntimeError("Could not identify the RGI7 glacier ID column.")

    # RGI7 has no nominal glaciers; only filter when an RGI6-style status field exists
    status_column = next((c for c in ("status", "Status") if c in rgidf.columns), None)
    if status_column is not None:
        nominal = rgidf[status_column].astype(str).str.strip().isin(["2", "9"])
        print("Nominal/unassigned glaciers removed:", int(nominal.sum()))
        rgidf = rgidf.loc[~nominal].copy()

    if MAX_GLACIERS is not None:
        rgidf = rgidf.head(MAX_GLACIERS).copy()
        print("TEST MODE: using", len(rgidf), "glaciers")
    print("Glaciers to model:", len(rgidf))
    if len(rgidf) == 0:
        raise RuntimeError("No glaciers in this region.")
    return rgidf

In [ ]:
# ======================================================================
# 5. PREPARE GLACIERS: DIRECTORIES, MASKS, CRU CLIMATE, VAS CALIBRATION (function)
# ======================================================================
#
# RGI-TOPO provides the official OGGM level-1 glacier directories for RGI7.
# The VAS calibration uses the RGI6/CRU reference t* table, applied spatially to RGI7.

PREPRO_BASE_URL = ("https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/"
                   "rgitopo/2026.1/all_dems/")

_REF_DF = None


def get_ref_table():
    global _REF_DF
    if _REF_DF is None:
        ref_table_file = vascaling.get_ref_tstars_filepath(VAS_REFERENCE_TSTARS_FILE)
        _REF_DF = pd.read_csv(ref_table_file)
        print("VAS reference table:", ref_table_file, f"({len(_REF_DF)} reference glaciers)")
        missing = {"lon", "lat", "tstar", "bias"} - set(_REF_DF.columns)
        if missing:
            raise RuntimeError(f"The VAS reference table is missing columns: {missing}")
    return _REF_DF


def run_missing(task, gdirs, filename, **kwargs):
    """Run an entity task only for glaciers whose output file is missing."""
    if RECOMPUTE_BASE:
        todo = list(gdirs)
    else:
        todo = [g for g in gdirs if not os.path.exists(g.get_filepath(filename))]
    print(f"{task.__name__}: {len(gdirs) - len(todo)} done already, {len(todo)} to run")
    if todo:
        workflow.execute_entity_task(task, todo, **kwargs)


def init_gdirs(rgidf):
    """Glacier directories of the current region.

    A restart must REUSE the existing folders: re-initialising from the preprocessed
    files overwrites the results of an earlier, interrupted run. So the first run
    downloads/extracts the folders and writes a marker; later runs only reopen them.
    """
    marker = WORKDIR / "gdirs_initialized.flag"
    cfg.PARAMS["border"] = 10
    if marker.exists() and not RECOMPUTE_BASE:
        try:
            info = json.loads(marker.read_text())
        except Exception:                       # older plain flag file
            info = {"max_glaciers": None}
        if info.get("max_glaciers") != MAX_GLACIERS:
            raise RuntimeError(
                f"{WORKDIR} was initialised with MAX_GLACIERS={info.get('max_glaciers')}, "
                f"but MAX_GLACIERS={MAX_GLACIERS} now. Use another OGGM_WORK_ROOT (e.g. for "
                "test runs), delete this region folder, or set RECOMPUTE_BASE = True.")
        print("Re-opening existing glacier directories (no re-initialisation).")
        return workflow.init_glacier_directories(rgidf, reset=False)

    gdirs = workflow.init_glacier_directories(
        rgidf,
        from_prepro_level=1,
        prepro_border=10,
        prepro_rgi_version=RGI_VERSION,
        prepro_base_url=PREPRO_BASE_URL,
    )
    marker.write_text(json.dumps({"max_glaciers": MAX_GLACIERS}))
    return gdirs


from oggm.shop import rgitopo


def select_dem(gdir):
    """The rgitopo 'all_dems' glacier folders contain one sub-folder per DEM source
    (NASADEM, COPDEM90, ...) and NO dem.tif in the main folder. Copy the chosen DEM there."""
    available = sorted(p.name for p in Path(gdir.dir).iterdir()
                       if p.is_dir() and (p / "dem.tif").exists())
    if not available:
        raise RuntimeError(f"no DEM sub-folders with dem.tif in {gdir.dir}")
    lat = float(gdir.cenlat)
    automatic = (["NASADEM", "COPDEM90", "COPDEM30"] if -56 < lat < 60
                 else ["COPDEM90", "COPDEM30", "NASADEM"])
    wanted = ([DEM_SOURCE] if DEM_SOURCE else []) + automatic
    choice = next((d for d in wanted if d in available), available[0])
    rgitopo.select_dem_from_dir(gdir, dem_source=choice, keep_dem_folders=False)


def ensure_dem(gdirs):
    """Select a DEM for every glacier that has no dem.tif yet (restart-safe)."""
    todo = [g for g in gdirs if not os.path.exists(g.get_filepath("dem"))]
    print(f"select_dem: {len(gdirs) - len(todo)} done already, {len(todo)} to run")
    failed = []
    for i, g in enumerate(todo, start=1):
        try:
            select_dem(g)
        except Exception as exc:
            failed.append((g.rgi_id, f"{type(exc).__name__}: {exc}"))
        if i % 2000 == 0:
            print(f"  DEM selected for {i}/{len(todo)} glaciers")
    if failed:
        print(f"DEM selection failed for {len(failed)} glaciers:")
        summarize_errors(failed)


def prepare_glaciers(rgidf):
    gdirs = init_gdirs(rgidf)
    print("Glacier directories:", len(gdirs))

    ensure_dem(gdirs)     # rgitopo all_dems: pick the DEM (no dem.tif in the main folder)

    run_missing(tasks.glacier_masks, gdirs, "gridded_data")

    # gridded_data.nc is required later: rebuild it where it is still missing
    broken = [g for g in gdirs if not os.path.exists(g.get_filepath("gridded_data"))]
    if broken:
        print(f"Rebuilding gridded_data for {len(broken)} glaciers...")
        for g in broken:
            try:
                tasks.glacier_masks(g, reset=True)
            except Exception as exc:
                print(f"  {g.rgi_id}: {exc}")

    # CRU climate: forcing of the reconstruction and reference of the VAS calibration
    run_missing(tasks.process_climate_data, gdirs, "climate_historical",
                y0=MODEL_START_YEAR, y1=CRU_RECON_END_YEAR)

    ref_df = get_ref_table()
    ref_df.to_csv(WORKDIR / "vas_ref_tstars_rgi7_spatial_cru.csv", index=False)
    run_missing(vascaling.local_t_star, gdirs, "vascaling_mustar", ref_df=ref_df)

    # keep only successfully calibrated glaciers
    records, valid_gdirs, failed_calibration = [], [], []
    for gdir in gdirs:
        try:
            cal = gdir.read_json("vascaling_mustar")
        except Exception:
            failed_calibration.append(gdir.rgi_id)
            continue
        rec = {"rgi_id": gdir.rgi_id,
               "t_star": cal.get("t_star", np.nan),
               "bias": cal.get("bias", np.nan),
               "mu_star": cal.get("mu_star", np.nan)}
        records.append(rec)
        if np.isfinite([rec["t_star"], rec["bias"], rec["mu_star"]]).all():
            valid_gdirs.append(gdir)
        else:
            failed_calibration.append(gdir.rgi_id)

    pd.DataFrame(records).to_csv(WORKDIR / "vas_calibration_rgi70.csv", index=False)
    pd.DataFrame({"rgi_id": failed_calibration}).to_csv(WORKDIR / "failed_vas_calibration.csv", index=False)
    print(f"VAS calibration: {len(valid_gdirs)} valid, {len(failed_calibration)} failed")

    if not valid_gdirs:
        raise RuntimeError("No glaciers passed the VAS calibration.")
    return valid_gdirs

In [ ]:
# ======================================================================
# 6. RESULT HELPERS (functions only)
# ======================================================================
# column name -> (model_diagnostics variable, unit factor)
DIAG_VARS = {
    "volume_km3": ("volume_m3", 1e-9),
    "area_km2": ("area_m2", 1e-6),
    "length_km": ("length_m", 1e-3),
    "specific_mb_mmwe": ("spec_mb", 1.0),
}


def read_diagnostics(gdir, suffix):
    with xr.open_dataset(gdir.get_filepath("model_diagnostics", filesuffix=suffix)) as ds:
        data = {"year": np.asarray(ds["time"].values, dtype=float).astype(int)}
        for column, (variable, factor) in DIAG_VARS.items():
            if variable in ds:
                data[column] = ds[variable].values * factor
    df = pd.DataFrame(data)
    df["rgi_id"] = gdir.rgi_id
    return df


def collect_results(gdirs, suffix):
    """Read the yearly diagnostics of all glaciers that have them."""
    frames, failed = [], []
    for i, gdir in enumerate(gdirs, start=1):
        try:
            frames.append(read_diagnostics(gdir, suffix))
        except Exception as exc:
            failed.append((gdir.rgi_id, str(exc)))
        if i % 500 == 0:
            print(f"  read {i}/{len(gdirs)} glaciers")
    if not frames:
        return pd.DataFrame(), failed
    df = pd.concat(frames, ignore_index=True).sort_values(["rgi_id", "year"]).reset_index(drop=True)
    return df, failed


def aggregate_region(glacier_df):
    """Regional sums. Length is the SUM of glacier lengths (total length)."""
    return (glacier_df.groupby("year", as_index=False)
            .agg(volume_km3=("volume_km3", "sum"),
                 area_km2=("area_km2", "sum"),
                 length_km=("length_km", "sum"))
            .sort_values("year").reset_index(drop=True))


def add_mass_and_sle(df, volume_ref_km3):
    """Mass loss and sea-level equivalent relative to a reference volume."""
    df = df.copy()
    loss = volume_ref_km3 - df["volume_km3"]
    df["volume_loss_km3"] = loss
    df["mass_loss_Gt"] = loss * RHO_ICE / RHO_WATER          # 1 km3 water = 1 Gt
    df["sea_level_change_mm"] = df["mass_loss_Gt"] / OCEAN_AREA_KM2 * 1e6
    df["annual_mass_change_Gt_yr"] = df["mass_loss_Gt"].diff()   # positive = loss
    df["annual_sle_mm_yr"] = df["sea_level_change_mm"].diff()
    return df


def volume_at(df, year):
    return float(df.loc[df["year"] == year, "volume_km3"].iloc[0])


def make_regional_hist(glacier_df):
    """Regional series with changes relative to MODEL_START_YEAR."""
    regional = aggregate_region(glacier_df)
    start = regional.loc[regional["year"] == MODEL_START_YEAR]
    if start.empty:
        raise RuntimeError(f"No results for {MODEL_START_YEAR}.")
    start = start.iloc[0]
    regional["volume_change_km3"] = regional["volume_km3"] - start["volume_km3"]
    regional["area_change_km2"] = regional["area_km2"] - start["area_km2"]
    regional["length_change_km"] = regional["length_km"] - start["length_km"]
    return add_mass_and_sle(regional, float(start["volume_km3"]))


In [ ]:
# ======================================================================
# 7. CRU RECONSTRUCTION (function)
# ======================================================================
def run_cru_reconstruction(gdirs):
    """VAS run MODEL_START_YEAR -> CRU_RECON_END_YEAR forced by CRU.
    Returns (gdirs that ran successfully, regional series, glacier series)."""
    print(f"\nVAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (CRU)")

    if RECOMPUTE_BASE:
        todo = list(gdirs)
    else:
        todo = [g for g in gdirs if not os.path.exists(
            g.get_filepath("model_diagnostics", filesuffix=CRU_HIST_SUFFIX))]
    print(f"{len(gdirs) - len(todo)} done already, {len(todo)} to run")

    if todo:
        workflow.execute_entity_task(
            vascaling.run_historic_from_climate_data,
            todo,
            ys=MODEL_START_YEAR,
            ye=CRU_RECON_END_YEAR,
            output_filesuffix=CRU_HIST_SUFFIX,
            climate_filename="climate_historical",
        )

    glacier_df, failed_hist = collect_results(gdirs, CRU_HIST_SUFFIX)
    if glacier_df.empty:
        raise RuntimeError("No successful historical VAS runs.")

    ok_ids = set(glacier_df["rgi_id"])
    pd.DataFrame({"rgi_id": [g.rgi_id for g in gdirs if g.rgi_id not in ok_ids]}).to_csv(
        WORKDIR / "failed_historical_rgi70.csv", index=False)
    gdirs = [g for g in gdirs if g.rgi_id in ok_ids]
    print(f"Reconstruction runs: {len(gdirs)} successful, {len(failed_hist)} failed")
    summarize_errors(failed_hist)

    regional = make_regional_hist(glacier_df)
    save_glacier_results(glacier_df)
    regional.to_csv(
        WORKDIR / f"regional_results_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}.csv", index=False)

    last = regional.loc[regional["year"] == regional["year"].max()].iloc[0]
    print(f"Volume {MODEL_START_YEAR}: {volume_at(regional, MODEL_START_YEAR):.3f} km3 | "
          f"{int(last['year'])}: {last['volume_km3']:.3f} km3 | "
          f"SLE: {last['sea_level_change_mm']:.5f} mm")
    return gdirs, regional, glacier_df


def save_glacier_results(glacier_df):
    """Per-glacier time series, in the format chosen by SAVE_GLACIER_RESULTS."""
    if not SAVE_GLACIER_RESULTS:
        print("Per-glacier results not saved (SAVE_GLACIER_RESULTS = False).")
        return
    stem = WORKDIR / f"glacier_results_hist{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}"
    # remove files from an earlier run with another format, so nothing is stored twice
    for ext in (".parquet", ".csv.gz", ".csv"):
        stem.with_name(stem.name + ext).unlink(missing_ok=True)

    fmt = SAVE_GLACIER_RESULTS
    if fmt == "parquet":
        small = glacier_df.copy()
        for col in small.select_dtypes("float64").columns:
            small[col] = small[col].astype("float32")
        small["year"] = small["year"].astype("int16")
        try:
            small.to_parquet(stem.with_name(stem.name + ".parquet"), index=False, compression="zstd")
        except ImportError:
            print("pyarrow is not installed: saving gzip-compressed CSV instead.")
            fmt = "csv.gz"
        except Exception:
            stem.with_name(stem.name + ".parquet").unlink(missing_ok=True)
            small.to_parquet(stem.with_name(stem.name + ".parquet"), index=False)
    if fmt == "csv.gz":
        glacier_df.to_csv(stem.with_name(stem.name + ".csv.gz"), index=False, float_format="%.6g")
    elif fmt == "csv":
        glacier_df.to_csv(stem.with_name(stem.name + ".csv"), index=False)


def clean_glacier_dirs():
    """Delete the OGGM glacier folders of the current region (big, only needed while it runs)."""
    folder = WORKDIR / "per_glacier"
    if not folder.exists():
        return
    try:
        size_gb = sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e9
        shutil.rmtree(folder)
        print(f"Cleaned glacier folders of this region ({size_gb:.2f} GB freed).")
    except Exception as exc:
        print(f"Could not clean glacier folders: {exc}")


def summarize_errors(errors, limit=3):
    """errors: list of (rgi_id, message). Print the most common messages."""
    if not errors:
        return
    counts = pd.Series([m for _, m in errors]).value_counts()
    for message, n in counts.head(limit).items():
        print(f"  {n} x {message[:200]}")


In [ ]:
# ======================================================================
# 8. REGIONAL CRU CLIMATE SERIES (function)
# ======================================================================
#
# CRU climate of the glacier grid cells (as stored in climate_historical),
# weighted by RGI glacier area. A diagnostic CSV only; it feeds no calculation.

def weighted_mean_by_year(df, column):
    d = df[np.isfinite(df[column]) & np.isfinite(df["w"]) & (df["w"] > 0)]
    return (d[column] * d["w"]).groupby(d["year"]).sum() / d["w"].groupby(d["year"]).sum()


def write_cru_climate(gdirs):
    ref_start, ref_end = int(ANOMALY_REFERENCE_PERIOD[0]), int(ANOMALY_REFERENCE_PERIOD[1])

    frames = []
    for gdir in gdirs:
        climate_file = gdir.get_filepath("climate_historical")
        if not os.path.exists(climate_file):
            continue
        try:
            with xr.open_dataset(climate_file) as ds:
                df = pd.DataFrame({"year": ds["time"].dt.year.values,
                                   "temp": ds["temp"].values,
                                   "prcp": ds["prcp"].values})
        except Exception as exc:
            print(f"Could not read CRU for {gdir.rgi_id}: {exc}")
            continue
        annual = df.groupby("year").agg(temp=("temp", "mean"), prcp=("prcp", "sum")).reset_index()
        annual["w"] = float(gdir.rgi_area_km2)
        frames.append(annual)

    if not frames:
        print("No CRU climate files found; regional CRU series skipped.")
        return
    cru_all = pd.concat(frames, ignore_index=True)

    cru_regional = pd.DataFrame({
        "cru_temperature_C": weighted_mean_by_year(cru_all, "temp"),
        "cru_precipitation_mm": weighted_mean_by_year(cru_all, "prcp"),
    }).reset_index()

    ref = cru_regional[cru_regional["year"].between(ref_start, ref_end)]
    cru_regional["cru_temperature_anomaly_C"] = (
        cru_regional["cru_temperature_C"] - ref["cru_temperature_C"].mean())
    cru_regional["cru_precipitation_anomaly_percent"] = (
        cru_regional["cru_precipitation_mm"] / ref["cru_precipitation_mm"].mean() - 1) * 100

    cru_regional.to_csv(
        WORKDIR / f"regional_cru_climate_{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}.csv", index=False)
    print("Regional CRU climate series saved.")

In [ ]:
# ======================================================================
# 9. PLOTS (function)
# ======================================================================
RECON_COLOR = "black"     # CRU reconstruction (IPCC-style neutral colour for observations/reconstructions)

PLOTS = [
    # column, y label, file key, draw zero line
    ("volume_km3", "Glacier volume (km³)", "volume", False),
    ("area_km2", "Glacier area (km²)", "area", False),
    ("length_km", "Sum of glacier lengths (km)", "length", False),
    ("mass_loss_Gt", f"Cumulative glacier mass loss since {MODEL_START_YEAR} (Gt)", "mass_loss", True),
    ("sea_level_change_mm", f"Sea-level equivalent relative to {MODEL_START_YEAR} (mm)", "sea_level", True),
]


def plot_reconstruction(column, ylabel, key, zero, reconstruction):
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(reconstruction["year"], reconstruction[column], color=RECON_COLOR,
            linewidth=2, label="VAS reconstruction (CRU)")
    if zero:
        ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(f"RGI70-{RGI_REGION:02d} - {REGION_NAME}\n"
                 f"VAS reconstruction {MODEL_START_YEAR}-{CRU_RECON_END_YEAR} (CRU-forced)")
    ax.set_xlim(MODEL_START_YEAR, CRU_RECON_END_YEAR)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(WORKDIR / f"{REGION_SLUG}_{key}_{MODEL_START_YEAR}_{CRU_RECON_END_YEAR}_cru.png", dpi=300)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


def plot_all(reconstruction):
    for column, ylabel, key, zero in PLOTS:
        plot_reconstruction(column, ylabel, key, zero, reconstruction)


In [ ]:
# ======================================================================
# 10. SUMMARY TABLE (function)
# ======================================================================
def make_summary(reconstruction, n_glaciers, n_calibrated, n_run_ok):
    first = reconstruction.iloc[0]
    last = reconstruction.iloc[-1]
    summary = pd.DataFrame([{
        "RGI_region": RGI_REGION,
        "region_name": REGION_NAME,
        "n_glaciers": n_glaciers,
        "n_calibrated": n_calibrated,
        "n_reconstructed": n_run_ok,
        "start_year": int(first["year"]),
        "end_year": int(last["year"]),
        "volume_start_km3": first["volume_km3"],
        "volume_end_km3": last["volume_km3"],
        "volume_loss_km3": first["volume_km3"] - last["volume_km3"],
        "area_start_km2": first["area_km2"],
        "area_end_km2": last["area_km2"],
        "mass_loss_Gt": last["mass_loss_Gt"],
        "sea_level_mm": last["sea_level_change_mm"],
        "lost_volume_percent": 100.0 * (1.0 - last["volume_km3"] / first["volume_km3"])
        if first["volume_km3"] > 0 else np.nan,
    }])
    summary.to_csv(WORKDIR / "reconstruction_summary.csv", index=False)
    print("\n" + "=" * 60)
    print(f"SUMMARY RGI70-{RGI_REGION:02d} {REGION_NAME}")
    print("=" * 60)
    print(summary.T.to_string(header=False))
    return summary


In [ ]:
# ======================================================================
# 11. ONE COMPLETE REGION (function)
# ======================================================================
def region_signature():
    """Settings that decide whether a finished region can be skipped."""
    return json.loads(json.dumps({
        "rgi": RGI_VERSION,
        "mode": "CRU_RECONSTRUCTION",
        "start": MODEL_START_YEAR, "end": CRU_RECON_END_YEAR,
        "max_glaciers": MAX_GLACIERS,
    }))


def run_region(region):
    """Complete CRU reconstruction of one RGI 7.0 region. Returns a status dict."""
    set_region(region)
    print("\n" + "#" * 70)
    print(f"# REGION {region:02d}: {REGION_NAME}")
    print("#" * 70)

    marker = WORKDIR / "region_done_cru.json"
    signature = region_signature()
    if SKIP_FINISHED_REGIONS and not RECOMPUTE_BASE and marker.exists():
        try:
            same = json.load(open(marker)) == signature
        except Exception:
            same = False
        if same:
            print("Already finished with these settings: skipped.")
            summary_file = WORKDIR / "reconstruction_summary.csv"
            summary = pd.read_csv(summary_file) if summary_file.exists() else pd.DataFrame()
            return {"status": "skipped", "n_glaciers": np.nan, "n_calibrated": np.nan,
                    "n_reconstructed": np.nan, "summary": summary}

    init_oggm()
    rgidf = load_region_glaciers()
    n_glaciers = len(rgidf)

    gdirs = prepare_glaciers(rgidf)
    n_calibrated = len(gdirs)

    gdirs, reconstruction, _ = run_cru_reconstruction(gdirs)
    n_run_ok = len(gdirs)

    write_cru_climate(gdirs)
    plot_all(reconstruction)
    summary = make_summary(reconstruction, n_glaciers, n_calibrated, n_run_ok)

    json.dump(signature, open(marker, "w"))
    if CLEAN_GLACIER_DIRS:
        clean_glacier_dirs()
    return {"status": "ok", "n_glaciers": n_glaciers, "n_calibrated": n_calibrated,
            "n_reconstructed": n_run_ok, "summary": summary}


In [ ]:
# ======================================================================
# 12. RUN - EDIT THE SWITCH HERE
# ======================================================================
# ##########################################################################
# #                                                                        #
# #   RUN - the only cell that starts calculations. Run it once and wait.  #
# #                                                                        #
# ##########################################################################

RUN_ALL_REGIONS = True                 # True : all regions in REGIONS_TO_RUN, one after the other
                                       # False: only the single region RGI_REGION (config cell)
REGIONS_TO_RUN = list(range(1, 20))    # 1-19 = all RGI 7.0 regions; or e.g. [8, 11]
CONTINUE_AFTER_REGION_ERROR = True     # True: log a failing region and go on with the next one
SHOW_PLOTS = False                     # True: also show the plots in the notebook (many regions = many plots)

regions = REGIONS_TO_RUN if RUN_ALL_REGIONS else [RGI_REGION]
print(f"Regions to run ({len(regions)}): {regions}")
if MAX_GLACIERS is not None:
    print(f"WARNING: MAX_GLACIERS = {MAX_GLACIERS} -> TEST MODE in every region!")

def preflight(hosts):
    """Once, before the region loop: is the internet reachable, and does OGGM start?
    (A failure here is not region-specific, so there is no point trying 19 regions.)"""
    import urllib.request, urllib.error
    print("Network check (only needed for files that are not in the download cache yet):")
    for name, url in hosts:
        try:
            with urllib.request.urlopen(url, timeout=20) as resp:
                status = f"reachable (HTTP {resp.status})"
        except urllib.error.HTTPError as exc:
            status = f"reachable (HTTP {exc.code})"
        except Exception as exc:
            status = f"NOT REACHABLE ({type(exc).__name__}: {exc})"
        print(f"  {name:46s} {status}")
    try:
        init_oggm()
    except Exception as exc:
        raise RuntimeError(
            "OGGM could not start, so no region can run. This usually means it could not "
            "download its sample data (GitHub) because this machine/node has no internet access: "
            "see the network check above. Fix the connection or copy the OGGM download cache "
            "to this machine, then run again.") from exc


preflight([("GitHub (OGGM sample data)", "https://github.com"),
           ("OGGM cluster (glacier directories, CRU)", "https://cluster.klima.uni-bremen.de/~oggm/")])

region_log, region_summaries = [], []
t_all = time.time()


def save_progress():
    pd.DataFrame(region_log).to_csv(WORK_ROOT / "all_regions_cru_log.csv", index=False)
    if region_summaries:
        pd.concat(region_summaries, ignore_index=True).to_csv(
            WORK_ROOT / "all_regions_reconstruction.csv", index=False)


try:
    for region in regions:
        t0 = time.time()
        row = {"region": region, "region_name": RGI_REGION_NAMES.get(region, "?")}
        try:
            out = run_region(region)
            row.update({k: v for k, v in out.items() if k != "summary"})
            if len(out["summary"]):
                region_summaries.append(out["summary"])
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            row.update({"status": "failed", "error": f"{type(exc).__name__}: {exc}"})
            print(f"\nREGION {region:02d} FAILED: {row['error']}")
            import traceback; traceback.print_exc()     # full error with line numbers
            if not CONTINUE_AFTER_REGION_ERROR:
                region_log.append({**row, "minutes": round((time.time() - t0) / 60, 1)})
                raise
        row["minutes"] = round((time.time() - t0) / 60, 1)
        region_log.append(row)

        save_progress()      # progress is saved after every region
        plt.close("all")
finally:
    if region_log:
        save_progress()

print("\n" + "=" * 70)
print(f"ALL DONE in {(time.time() - t_all) / 60:.1f} minutes")
print("=" * 70)
log_df = pd.DataFrame(region_log)
print(log_df[[c for c in ["region", "region_name", "status", "n_glaciers", "n_calibrated",
                          "n_reconstructed", "minutes"] if c in log_df]]
      .to_string(index=False))

if region_summaries:
    all_df = pd.concat(region_summaries, ignore_index=True)
    total = all_df[["volume_start_km3", "volume_end_km3", "volume_loss_km3",
                    "mass_loss_Gt", "sea_level_mm"]].sum()
    print(f"\nTotal over {len(all_df)} regions: volume {total['volume_start_km3']:.0f} -> "
          f"{total['volume_end_km3']:.0f} km3, mass loss {total['mass_loss_Gt']:.0f} Gt, "
          f"sea-level equivalent {total['sea_level_mm']:.2f} mm")
print("\nLog           :", WORK_ROOT / "all_regions_cru_log.csv")
print("Reconstruction:", WORK_ROOT / "all_regions_reconstruction.csv")
